# 시나리오1 보조 — 장표 요약 사건과 오프라인 필터

교안 127페이지의 요약 사건과 123~124페이지 조건을 파일만으로 연습합니다. 이 노트북은 3주 CSV와 별개입니다. 실제 AWS 수집 구성·Metric Filter 배포·알림 발송을 수행하는 것은 아닙니다.

Colab에서 이 노트북을 열고 왼쪽 파일 창에 `장표127_요약사건.json`, `필터연습_가상입력.json`을 올립니다. 로컬은 같은 폴더에 둡니다. 날짜·시간대가 없는 장표 사건에는 임의의 날짜를 붙이지 않습니다.

In [ ]:
from pathlib import Path
import json, ipaddress
import pandas as pd
from IPython.display import display
search=[Path.cwd(),Path.cwd()/'학생용',Path.cwd().parent/'학생용',Path('/content')]
data_dir=next((p for p in search if (p/'장표127_요약사건.json').is_file() and (p/'필터연습_가상입력.json').is_file()),None)
if data_dir is None:raise FileNotFoundError('두 JSON 파일을 노트북 옆 또는 Colab 파일 창에 올리세요.')
summary=json.loads((data_dir/'장표127_요약사건.json').read_text(encoding='utf-8'))
events=pd.DataFrame(summary['rows'])
events['order_time']=pd.to_timedelta(events.clock)
office=ipaddress.ip_network('198.51.100.0/24')
events['office_source']=events.source_ip.map(lambda value:ipaddress.ip_address(value) in office)
display(events[['slide_line','clock','event_label','actor','source_ip','result','notes']])
print('전사한 사건 요약 행:',len(events));print(summary['date_timezone'])

## 학생 조사

1. 사무실 안팎의 호출을 나누어 로그인 실패→성공 및 MFA 표기를 찾습니다.
2. 키 생성, 정책 연결, 새 사용자, 자료 읽기, 은폐 시도를 시각·slide_line으로 정리합니다.
3. DENIED가 막은 행동과 OK로 요약된 행동을 구분합니다.
4. 키 사용 이력·정책 원문·객체 응답 바이트가 없으므로 피해량, 새 키로 실행한 요청, 적용 권한 전체는 미확인으로 적습니다.
5. 마지막 두 조치로 support2까지 차단되었다고 말할 수 있는지 검토합니다.

주체 이름·사무실 IP는 출처 맥락이며 자동으로 정상 또는 동일 인간을 증명하지 않습니다. OK는 장표의 요약 표기이지 원시 CloudTrail 응답 검증은 아닙니다.

In [ ]:
# 조건과 집계를 직접 바꾸어 후보를 살펴보세요.
outside=events[~events.office_source].sort_values('order_time')
display(outside[['slide_line','clock','actor','event_label','result','notes']])
display(events.groupby(['actor','result']).size().rename('events').reset_index())
blocked=events[events.result.eq('DENIED')]
display(blocked[['slide_line','clock','event_label']])

## 오프라인 필터: 이름만 고르면 요구 조건이 빠지는가?

다음 입력은 필터 테스트용으로 새로 만든 작은 JSON입니다. 127페이지나 CSV의 미제공 필드를 보완하는 사건 증거가 아닙니다.

- Root 성공 로그인: ConsoleLogin AND Root AND responseElements.ConsoleLogin=Success. 응답이 없으면 성공으로 분류하지 않습니다.
- 야간 IAM 사용자 키 생성: CreateAccessKey AND IAMUser AND KST 21~06시. 실패 여부와 별개로 호출 시도를 셉니다.
- 5분 창에서 1회 이상이면 가상 경보로 표시합니다.

아래 코드를 수정해 IAMUser 조건 또는 야간 조건을 빼고 결과가 어떻게 바뀌는지 비교하세요. 이것은 로컬 판정 모형이며 CloudWatch 패턴 문법 전체를 구현한 엔진이 아닙니다.

In [ ]:
inputs=json.loads((data_dir/'필터연습_가상입력.json').read_text(encoding='utf-8'))
root_success=[r for r in inputs if r['eventName']=='ConsoleLogin' and r['userIdentity']['type']=='Root' and (r.get('responseElements') or {}).get('ConsoleLogin')=='Success']
key_inputs=pd.DataFrame([{'event_id':r['eventID'],'time':r['eventTime'],'event':r['eventName'],'kind':r['userIdentity']['type'],'error':r.get('errorCode','')} for r in inputs])
key_inputs['kst']=pd.to_datetime(key_inputs.time,utc=True).dt.tz_convert('Asia/Seoul')
broad=key_inputs[key_inputs.event.eq('CreateAccessKey')]
night=key_inputs.kst.dt.hour.ge(21)|key_inputs.kst.dt.hour.lt(7)
matched=key_inputs[key_inputs.event.eq('CreateAccessKey') & key_inputs.kind.eq('IAMUser') & night].copy()
matched['window']=matched.kst.dt.floor('5min')
alarms=matched.groupby('window').size().rename('count').reset_index()
alarms['alarm']=alarms['count'].ge(1)
print('Root 성공:',len(root_success),'/ API 이름만 고른 키 생성:',len(broad),'/ 요구 조건을 적용한 키 생성:',len(matched))
display(alarms)
print('MFA 누락·로그인 결과 누락을 성공으로 채우지 않습니다.')

## 제출

장표 사건 타임라인(근거 5개), 은폐 시도의 결과, 마지막 대응의 남은 범위, 필터 조건 변경 전후 표를 제출합니다.

실제 AWS 리소스 생성·SNS 수신은 별도 계정·권한·수집 환경이 필요한 선택 실습입니다. 이 오프라인 실습의 필수 단계에 넣지 않습니다. 따라서 외부 이벤트 도착을 기다리느라 멈추지 않습니다.